# QLoRA Fine-Tuning: Medical/Biomedical Abstract Summarizer

This notebook fine-tunes **`microsoft/Phi-3-mini-4k-instruct`** with **QLoRA** (4-bit quantization + LoRA adapters) on a subset of **`ccdv/pubmed-summarization`** (full PubMed article -> abstract) to produce a technical summarization model. This is the Summarizer Agent's backend for the Multi-Agent Medical Document Summarizer project.

**Run this on Google Colab with a free T4 GPU runtime** (`Runtime > Change runtime type > T4 GPU`). It also runs unmodified on Kaggle's free GPU quota.

## Why these choices
- **Base model — `microsoft/Phi-3-mini-4k-instruct` (3.8B, MIT license):** ungated on the Hugging Face Hub (no click-through license/approval needed, unlike Llama-3.2-3B-Instruct which requires per-account Meta license acceptance), well-documented QLoRA recipes, and small enough to fine-tune on a free 16GB T4.
- **Dataset — `ccdv/pubmed-summarization`:** long PubMed articles paired with their abstracts as reference summaries. No credentialing required (unlike MIMIC-III/IV). We use a *subset* (a few thousand examples), not the full ~120k-example dataset, so this fits in free-tier GPU time.
- **4-bit NF4 quantization (bitsandbytes) + LoRA adapters (peft):** lets us fine-tune a ~4B parameter model on a 16GB GPU by keeping the frozen base weights in 4-bit and only training small low-rank adapter matrices.

## What this notebook produces
1. A trained LoRA adapter (a few tens of MB) saved to `./phi3-mini-pubmed-qlora-adapter/`
2. Real ROUGE-1 / ROUGE-2 / ROUGE-L scores on a held-out test split, saved to `eval_results.json`
3. Example generations for manual inspection

Download the adapter folder (or push it to your own Hugging Face Hub repo, see the last cell) and copy it into `backend/models/adapter/` in the main project — that's what `backend/app/agents/summarizer_agent.py` loads at inference time.

## Step 1 — Install dependencies
Pinned to versions known to work together for QLoRA on Colab's CUDA 12.x images as of late 2025.

In [ ]:
!pip install -q -U "transformers>=4.44.0" "peft>=0.12.0" "bitsandbytes>=0.43.1" "accelerate>=0.33.0" "datasets>=2.20.0" "trl>=0.9.6" "evaluate>=0.4.2" "rouge_score" "sentencepiece" "absl-py"


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU detected. In Colab: Runtime > Change runtime type > T4 GPU, then Runtime > Restart session.")


## Step 2 — Config
All the knobs in one place so it's easy to see (and change) what was used.

In [ ]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"

# --- Dataset subset sizes (kept small deliberately to fit free-tier Colab GPU time budget) ---
N_TRAIN = 2000
N_VAL = 200
N_TEST = 200

# --- Sequence length budget ---
# Phi-3-mini-4k-instruct has a 4096-token context. We truncate source articles so that
# prompt + article + target summary comfortably fits, leaving room for generation at inference.
MAX_SOURCE_TOKENS = 1200
MAX_TARGET_TOKENS = 300
MAX_SEQ_LEN = 1792  # prompt template + truncated source + target, with margin

# --- LoRA hyperparameters ---
# r=16 / alpha=32 (2x rank, a common default) balances adapter capacity against overfitting risk
# on a ~2k-example training set. Targeting all attention + MLP projection matrices (not just
# q_proj/v_proj) gives the adapter more capacity to shift the model's summarization behavior,
# which matters more here than in typical instruction-tuning since we're changing domain + task.
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# --- Training hyperparameters ---
NUM_EPOCHS = 2
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8  # effective batch size = 2 * 8 = 16
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03

OUTPUT_DIR = "./phi3-mini-pubmed-qlora-adapter"

SYSTEM_PROMPT = (
    "You are a clinical documentation assistant. Read the biomedical article and write a "
    "precise, technical summary suitable for a clinician: preserve key diagnoses, procedures, "
    "medications, dosages, and findings. Do not add information that is not in the source."
)


## Step 3 — Load and prepare the dataset

In [ ]:
from datasets import load_dataset

raw = load_dataset("ccdv/pubmed-summarization", "document")
print(raw)

train_ds = raw["train"].shuffle(seed=SEED).select(range(N_TRAIN))
val_ds = raw["validation"].shuffle(seed=SEED).select(range(N_VAL))
test_ds = raw["test"].shuffle(seed=SEED).select(range(N_TEST))

print(train_ds[0]["article"][:500])
print("---ABSTRACT---")
print(train_ds[0]["abstract"][:500])


## Step 4 — Load the base model in 4-bit (QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",  # safest default across Colab's T4 (no flash-attn wheel there)
)
model.config.use_cache = False


## Step 5 — Attach LoRA adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## Step 6 — Format examples as chat-style prompts
We truncate the source article (by tokens) before formatting so the whole example fits within `MAX_SEQ_LEN`, and mask the loss on everything except the assistant's response (`DataCollatorForCompletionOnlyLM`) so the model is only trained to predict the summary, not to reproduce the prompt/article.

In [ ]:
def truncate_article(article: str, max_tokens: int) -> str:
    ids = tokenizer.encode(article, add_special_tokens=False)
    if len(ids) > max_tokens:
        ids = ids[:max_tokens]
    return tokenizer.decode(ids, skip_special_tokens=True)


def format_example(example):
    article = truncate_article(example["article"], MAX_SOURCE_TOKENS)
    abstract = example["abstract"].strip()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Summarize this article:\n\n{article}"},
        {"role": "assistant", "content": abstract},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}


train_formatted = train_ds.map(format_example, remove_columns=train_ds.column_names)
val_formatted = val_ds.map(format_example, remove_columns=val_ds.column_names)

print(train_formatted[0]["text"][:1500])


## Step 7 — Train with `SFTTrainer`

In [ ]:
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

# Phi-3 chat template renders the assistant turn as "<|assistant|>\n...". We mask loss on
# everything before this marker so the model only learns to generate the summary.
response_template = "<|assistant|>\n"
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

sft_config = SFTConfig(
    output_dir="./train_output",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    optim="paged_adamw_8bit",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="epoch",
    bf16=True,
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    data_collator=collator,
)

train_result = trainer.train()
print(train_result)


## Step 8 — Save the LoRA adapter

In [ ]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

import os
for f in os.listdir(OUTPUT_DIR):
    print(f, os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1e6, "MB")


## Step 9 — Real evaluation: ROUGE-1 / ROUGE-2 / ROUGE-L on the held-out test split
These numbers come from actually generating summaries with the fine-tuned model and comparing them to the reference abstracts with the standard `rouge_score` implementation. **Do not treat any number outside this cell's output as real** — copy the printed dict verbatim into `docs/RESULTS.md` in the main project.

In [ ]:
import evaluate
from tqdm.auto import tqdm

rouge = evaluate.load("rouge")
model.eval()
model.config.use_cache = True

predictions, references = [], []

for example in tqdm(test_ds, desc="Generating test summaries"):
    article = truncate_article(example["article"], MAX_SOURCE_TOKENS)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Summarize this article:\n\n{article}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_TOKENS,
            do_sample=False,
            num_beams=1,
            pad_token_id=tokenizer.pad_token_id,
        )
    gen_text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    predictions.append(gen_text)
    references.append(example["abstract"].strip())

results = rouge.compute(predictions=predictions, references=references, use_stemmer=True)
print("ROUGE results on held-out test set (n=%d):" % len(test_ds))
for k, v in results.items():
    print(f"  {k}: {v:.4f}")

import json
with open("eval_results.json", "w") as f:
    json.dump({"n_test": len(test_ds), "rouge": results, "base_model": BASE_MODEL,
               "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "epochs": NUM_EPOCHS}, f, indent=2)
print("Saved eval_results.json")


## Step 10 — Spot-check a few generations by eye

In [ ]:
for i in range(3):
    print("=" * 80)
    print("REFERENCE:\n", references[i][:600])
    print("-" * 80)
    print("GENERATED:\n", predictions[i][:600])


## Step 11 — Get the adapter out of Colab
**Option A (recommended): push to your own Hugging Face Hub repo** — the backend can then load it directly by repo id, no manual file transfer needed.

**Option B:** zip and download, then unzip into `backend/models/adapter/` in the main project locally.

In [ ]:
# --- Option A: push to Hugging Face Hub (uncomment and fill in your username/token) ---
# from huggingface_hub import login
# login(token="hf_...")  # get a token with WRITE access from https://huggingface.co/settings/tokens
# HF_REPO_ID = "<your-hf-username>/phi3-mini-pubmed-qlora-adapter"
# trainer.model.push_to_hub(HF_REPO_ID)
# tokenizer.push_to_hub(HF_REPO_ID)
# print(f"Pushed to https://huggingface.co/{HF_REPO_ID}")

# --- Option B: zip for manual download ---
import shutil
shutil.make_archive("phi3-mini-pubmed-qlora-adapter", "zip", OUTPUT_DIR)
print("Created phi3-mini-pubmed-qlora-adapter.zip — download it from the Colab file browser (left sidebar).")

try:
    from google.colab import files
    files.download("phi3-mini-pubmed-qlora-adapter.zip")
    files.download("eval_results.json")
except ImportError:
    pass  # not running in Colab (e.g. Kaggle) — grab the files from the output panel instead
